## 6a — Update QV Answers (Subset Pipeline)

Same pipeline as `6.update_answers.ipynb`, but scoped to a question subset provided as a CSV.

**Before running:**
- Set `ANTHROPIC_API_KEY` in `.env`
- Place `ChronosAlign.parquet` in `./data/final/`
- Provide a subset CSV with columns: `question`, `answer`, `event_year`

**Configure these before running:**

| Variable | Cell | Description |
|----------|------|-------------|
| `df_sample` path | 2 | Your subset CSV |
| `UPDATE_YEARS` | 3 | Target years to generate answers for |
| JSONL path | 9 | Downloaded batch results (if not using live API) |
| Output path | 15 | Output parquet path |

A sample subset is provided for evaluation: `./sample_data/tour_de_france_2024.csv`


In [24]:
from prompt_config.config import user_prompt_rephrase
import os
import pandas as pd
from src.utils.file_utils import url_to_hash
from collections import defaultdict

import aiohttp
import nest_asyncio
import pandas as pd

from src.utils.wikipedia_extractor import WikipediaExtractor
from src.utils.batch_anthropic_5000_concurrent import get_pageviews_async

# --- Notebook parameters ---
# Calendar years for which to build update rows (event_year on each generated record,
# Wikipedia pageview range, and questions sent to the batch API).
UPDATE_YEARS = [2025, 2026]
SAMPLE_PATH = './sample_data/tour_de_france_2024.csv'
BASE_PATH = './data/final/ChronosAlign.parquet'

years = sorted({int(y) for y in UPDATE_YEARS})

In [ ]:
# Load the full benchmark dataset and a user-provided question subset file.
df = pd.read_parquet(BASE_PATH)

# Replace this path with any CSV/dataframe export that has at least:
# question, answer, event_year
df_sample = pd.read_csv(SAMPLE_PATH)
# df_sample = df.head()

required_cols = {'question', 'answer', 'event_year'}
missing_cols = required_cols - set(df_sample.columns)
if missing_cols:
    raise ValueError(f"df_sample is missing required columns: {sorted(missing_cols)}")

# Match on normalized question text to avoid whitespace mismatch issues.
sample_question_set = set(df_sample['question'].dropna().astype(str).str.strip())
qv_question_set = sample_question_set  # Backward-compatible alias used in downstream cells.
df_qv_matched = df[df['question'].astype(str).str.strip().isin(sample_question_set)].copy()

# Keep one row per (title, question) for quick sanity checks.
unique_questions = (
    df_qv_matched.groupby(['url', 'question'], as_index=False)['views']
    .sum()
    .reset_index(drop=True)
)

len(unique_questions), len(sample_question_set)

In [ ]:
df.count()

In [ ]:


# Build update inputs only for questions present in the provided sample.
df_tdf = df[df['question'].astype(str).str.strip().isin(qv_question_set)].copy()

extractor = WikipediaExtractor()
unique_wiki_urls = df_tdf['url'].dropna().unique()

# Fetch Wikipedia content once per URL (reused across all questions/years).
wiki_content_rows = []
for wiki_url in unique_wiki_urls:
    extracted_content = extractor.extract_wikipedia_content(wiki_url)
    wiki_content_rows.append({'url': wiki_url, 'content': extracted_content})
wiki_content_df = pd.DataFrame(wiki_content_rows)

# Template rows are unique (url, question) pairs, then content is attached.
template_questions = (
    df_tdf[['url', 'question']]
    .drop_duplicates()
    .merge(wiki_content_df[['url', 'content']], on='url', how='left')
    .reset_index(drop=True)
)
df_tdf_enriched = template_questions[['url', 'question', 'content']]

# Required schema for synthetic rows where no source year exists.
all_properties = [
    'event_year', 'event_date', 'subject', 'relationship', 'answer',
    'question',  'event', 'eventLabel', 'url','title', 'content', 'content_length',
    'wikipedia_url_id', 'views', 'new_question', 'categories',
    'relative_question', 'id',
]

nest_asyncio.apply()


def get_views_for_years_sync(wiki_url, target_years, base_url='https://en.wikipedia.org/wiki/'):
    """Return yearly pageview totals as {year: int|None} for one wiki URL."""
    import asyncio

    article_title = wiki_url.replace(' ', '_')
    url = f"{base_url}{article_title}"
    start_date = f"{min(target_years)}0101"
    end_date = f"{max(target_years)}1231"
    views_per_year = {year: None for year in target_years}

    async def fetch_async():
        async with aiohttp.ClientSession(headers={'User-Agent': 'Research Project'}) as session:
            return await get_pageviews_async(session, url, start_date, end_date)

    try:
        raw_results = asyncio.get_event_loop().run_until_complete(fetch_async())
        if raw_results:
            pv_df = pd.DataFrame(raw_results)
            pv_df['year'] = pv_df['timestamp'].astype(str).str[:4].astype(int)
            for year in target_years:
                total_year_views = pv_df.loc[pv_df['year'] == year, 'views'].sum()
                views_per_year[year] = int(total_year_views) if pd.notna(total_year_views) and total_year_views > 0 else None
    except Exception as exc:
        print(f"Error fetching views for {wiki_url}: {exc}")

    return views_per_year


records = []
wiki_url_to_years = defaultdict(set)

for _, row in df_tdf_enriched.iterrows():
    wiki_url = row['url']
    question_text = row['question']
    content_text = row.get('content')

    source_rows = df_tdf[(df_tdf['url'] == wiki_url) & (df_tdf['question'] == question_text)]

    for target_year in years:
        exact_row = source_rows[source_rows['event_year'] == target_year]

        if not exact_row.empty:
            entry = exact_row.iloc[0].to_dict()
        elif not source_rows.empty:
            available_years = source_rows['event_year'].dropna().astype(int)
            closest_year = available_years.iloc[(available_years - target_year).abs().argsort()].iloc[0]
            entry = source_rows[source_rows['event_year'] == closest_year].iloc[0].to_dict()
        else:
            entry = {k: None for k in all_properties}
            entry['url'] = wiki_url
            entry['question'] = question_text
            entry['event_date'] = ''
            entry['views'] = ''

        # Force re-answering for the target year while preserving other metadata.
        entry['answer'] = ''
        entry['event_year'] = target_year
        entry['content'] = content_text

        wiki_url_to_years[wiki_url].add(target_year)
        records.append(entry)

# Fetch pageview totals once per URL and populate all generated entries.
all_wiki_views = {
    wiki_url: get_views_for_years_sync(wiki_url, sorted(list(year_set)))
    for wiki_url, year_set in wiki_url_to_years.items()
}

for entry in records:
    wiki_url = entry['url']
    target_year = entry['event_year']
    entry['views'] = all_wiki_views.get(wiki_url, {}).get(target_year, entry.get('views', None))


In [ ]:
# Keep only fields needed by the batch payload.
import json

df_reduced = pd.DataFrame(records)[['event_year', 'answer', 'question', 'url', 'content']].copy()
df_reduced['url_hash'] = df_reduced['url'].apply(lambda value: url_to_hash(str(value)))

# Normalize content so groupby keys are hashable even when content is dict/list.
df_reduced['content_group_key'] = df_reduced['content'].apply(
    lambda value: value
    if isinstance(value, str)
    else json.dumps(value, sort_keys=True)
    if isinstance(value, (dict, list))
    else str(value)
)

df_reduced['qa_item'] = df_reduced.apply(
    lambda row: {
        'event_year': row['event_year'],
        'answer': row['answer'],
        'question': row['question'],
    },
    axis=1,
)

# One payload row per article content with all year-specific question objects.
df_grouped = (
    df_reduced.groupby(['url_hash', 'content_group_key'], as_index=False)
    .agg(questions=('qa_item', list))
    .rename(columns={'content_group_key': 'content'})
)

df_grouped

In [ ]:
from src.utils.ai_question_generator import AIQuestionGenerator
from prompt_config.config import system_prompt, user_prompt_update_questions

# Instantiate the AIQuestionGenerator
ai_gen = AIQuestionGenerator(provider="anthropic")  # or "openai" depending on your setup

# Prepare the contents for batch processing with questions using Anthropic
contents_for_batch = []
for idx, row in df_grouped.iterrows():
    questions = row['questions']
    content_text = row['content']
    url_hash = row['url_hash']
    # Each entry should contain the context/content and a list of questions
    contents_for_batch.append(
        {
            "content": content_text,
            "questions": questions,
            "custom_id": url_hash
        }
    )

# Generate the batch using Anthropic's batch with questions method
batch_response = ai_gen.generate_qa_batch_with_questions(
    contents=contents_for_batch,
    system_prompt=system_prompt,
    user_prompt=user_prompt_update_questions,  # Must have {content} and {questions}
    max_tokens=4000
)

In [ ]:
# Check the batch result
batch_results = ai_gen.get_batch_results(batch_response["batch_id"])
batch_results

In [25]:
## We manually download from anthropic and save the jsonl file.
from src.utils.helper import extract_messages_from_jsonl
msg_0 = extract_messages_from_jsonl("./data/processed/batch_results_msgbatch_01P2iby6M2SwJkC5GE41Dg1p.jsonl")

Total errors: 0


In [26]:


# Normalize batch output to a DataFrame and align id naming with downstream joins.
msg_0_df = pd.DataFrame(msg_0) if isinstance(msg_0, list) else msg_0.copy()
msg_0_df = msg_0_df.rename(columns={'id': 'wikipedia_url_id'})

msg_0_df

,event_year,answer,question,wikipedia_url_id
0,2025,Jonathan Milan,who won the points classification in the Tour ...,3q4pc9pvduiv92hpw2yg7ar7mmvfjqne9ldqm5xq1tir17...
1,2026,,who won the points classification in the Tour ...,3q4pc9pvduiv92hpw2yg7ar7mmvfjqne9ldqm5xq1tir17...
2,2025,Tadej Pogačar,who finished second in the points classificati...,3q4pc9pvduiv92hpw2yg7ar7mmvfjqne9ldqm5xq1tir17...
3,2026,,who finished second in the points classificati...,3q4pc9pvduiv92hpw2yg7ar7mmvfjqne9ldqm5xq1tir17...
4,2025,Biniam Girmay,who finished third in the points classificatio...,3q4pc9pvduiv92hpw2yg7ar7mmvfjqne9ldqm5xq1tir17...
5,2026,,who finished third in the points classificatio...,3q4pc9pvduiv92hpw2yg7ar7mmvfjqne9ldqm5xq1tir17...
6,2025,Tadej Pogačar,who was the winner of the tour de france?,sl6egxw7xt1d4qnjbsa04lqnklntkyr03bd8rgv225k72tqxs
7,2026,,who was the winner of the tour de france?,sl6egxw7xt1d4qnjbsa04lqnklntkyr03bd8rgv225k72tqxs
8,2025,"3,320 km (2,060 mi)",what was the distance of the tour de france?,sl6egxw7xt1d4qnjbsa04lqnklntkyr03bd8rgv225k72tqxs
9,2026,,what was the distance of the tour de france?,sl6egxw7xt1d4qnjbsa04lqnklntkyr03bd8rgv225k72tqxs


In [27]:
# Join batch outputs to record metadata using (question, event_year).
df_records = pd.DataFrame(records)

record_extra_cols = [
    col for col in df_records.columns
    if col not in msg_0_df.columns and col not in ['question', 'event_year']
]

expand_to_full_df = msg_0_df.merge(
    df_records[['question', 'event_year'] + record_extra_cols],
    on=['question', 'event_year'],
    how='left',
    suffixes=('', '_from_records'),
)

# Prefer batch-provided fields, but backfill from template records when missing.
for column_name in ['subject', 'relationship', 'event_date']:
    source_column = f'{column_name}_from_records'
    if source_column in expand_to_full_df.columns:
        expand_to_full_df[column_name] = expand_to_full_df[column_name].combine_first(expand_to_full_df[source_column])
        expand_to_full_df = expand_to_full_df.drop(columns=[source_column])

expand_to_full_df['event_year'] = pd.to_numeric(expand_to_full_df['event_year'], errors='coerce')


def normalize_answer(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    return [str(value)]


expand_to_full_df['answer'] = expand_to_full_df['answer'].apply(normalize_answer)

expand_to_full_df


,event_year,answer,question,wikipedia_url_id,event_date,subject,relationship,event,eventLabel,url,title,content,content_length,views,categories,relative_question,id
0,2025,[Jonathan Milan],who won the points classification in the Tour ...,3q4pc9pvduiv92hpw2yg7ar7mmvfjqne9ldqm5xq1tir17...,None,[Tour de France Points Classification],winner,None,None,https://en.wikipedia.org/wiki/Points_classific...,Points classification in the Tour de France,"{'success': True, 'url': 'https://en.wikipedia...",19458,89894,sport,Who won the points classification in the most ...,43654
1,2026,[],who won the points classification in the Tour ...,3q4pc9pvduiv92hpw2yg7ar7mmvfjqne9ldqm5xq1tir17...,None,[Tour de France Points Classification],winner,None,None,https://en.wikipedia.org/wiki/Points_classific...,Points classification in the Tour de France,"{'success': True, 'url': 'https://en.wikipedia...",19458,10797,sport,Who won the points classification in the most ...,43654
2,2025,[Tadej Pogačar],who finished second in the points classificati...,3q4pc9pvduiv92hpw2yg7ar7mmvfjqne9ldqm5xq1tir17...,None,[Tour de France Points Classification],second_place,None,None,https://en.wikipedia.org/wiki/Points_classific...,Points classification in the Tour de France,"{'success': True, 'url': 'https://en.wikipedia...",19458,89894,sport,Who most recently finished second in the point...,43654
3,2026,[],who finished second in the points classificati...,3q4pc9pvduiv92hpw2yg7ar7mmvfjqne9ldqm5xq1tir17...,None,[Tour de France Points Classification],second_place,None,None,https://en.wikipedia.org/wiki/Points_classific...,Points classification in the Tour de France,"{'success': True, 'url': 'https://en.wikipedia...",19458,10797,sport,Who most recently finished second in the point...,43654
4,2025,[Biniam Girmay],who finished third in the points classificatio...,3q4pc9pvduiv92hpw2yg7ar7mmvfjqne9ldqm5xq1tir17...,None,[Tour de France Points Classification],third_place,None,None,https://en.wikipedia.org/wiki/Points_classific...,Points classification in the Tour de France,"{'success': True, 'url': 'https://en.wikipedia...",19458,89894,sport,Who most recently finished third in the points...,43654
5,2026,[],who finished third in the points classificatio...,3q4pc9pvduiv92hpw2yg7ar7mmvfjqne9ldqm5xq1tir17...,None,[Tour de France Points Classification],third_place,None,None,https://en.wikipedia.org/wiki/Points_classific...,Points classification in the Tour de France,"{'success': True, 'url': 'https://en.wikipedia...",19458,10797,sport,Who most recently finished third in the points...,43654
6,2025,[Tadej Pogačar],who was the winner of the tour de france?,sl6egxw7xt1d4qnjbsa04lqnklntkyr03bd8rgv225k72tqxs,None,[Tour de France],winner,http://www.wikidata.org/entity/Q462671,list of Tour de France winners,https://en.wikipedia.org/wiki/List_of_Tour_de_...,List of Tour de France general classification ...,"{'success': True, 'url': 'https://en.wikipedia...",21711,721944,sport,Who is the most recent winner of the tour de f...,7126
7,2026,[],who was the winner of the tour de france?,sl6egxw7xt1d4qnjbsa04lqnklntkyr03bd8rgv225k72tqxs,None,[Tour de France],winner,http://www.wikidata.org/entity/Q462671,list of Tour de France winners,https://en.wikipedia.org/wiki/List_of_Tour_de_...,List of Tour de France general classification ...,"{'success': True, 'url': 'https://en.wikipedia...",21711,88057,sport,Who is the most recent winner of the tour de f...,7126
8,2025,"[3,320 km (2,060 mi)]",what was the distance of the tour de france?,sl6egxw7xt1d4qnjbsa04lqnklntkyr03bd8rgv225k72tqxs,None,[Tour de France],distance,http://www.wikidata.org/entity/Q462671,list of Tour de France winners,https://en.wikipedia.org/wiki/List_of_Tour_de_...,List of Tour de France general classification ...,"{'success': True, 'url': 'https://en.wikipedia...",21711,721944,sport,What was the distance of the most recent tour ...,7126
9,2026,[],what was the distance of the tour de france?,sl6egxw7xt1d4qnjbsa04lqnklntkyr03bd8rgv225k72tqxs,Non

In [28]:
# Sanity check: each URL should have one row per requested target year.
event_year_counts = expand_to_full_df.groupby('wikipedia_url_id')['event_year'].nunique()
urls_with_missing_years = event_year_counts[event_year_counts < len(years)].index.tolist()

urls_with_missing_years

[]

In [29]:
df = pd.read_parquet('./data/final/ChronosAlign.parquet')

# More efficient upsert: concatenate and drop duplicates, keeping the last (updated) records
merge_keys = ['question', 'event_year']

# Validation: Check that updated records match old records (except for content, views, answer)
# Find records that exist in both dataframes (being updated)
old_records = df.set_index(merge_keys)
new_records = expand_to_full_df.set_index(merge_keys)

# Get common index (records being updated)
common_keys = old_records.index.intersection(new_records.index)

if len(common_keys) > 0:
    # Columns to exclude from comparison
    exclude_cols = {'content','views', 'answer'}
    
    # Get columns to compare (all columns except excluded ones)
    old_cols = set(old_records.columns) - exclude_cols
    new_cols = set(new_records.columns) - exclude_cols
    compare_cols = list(old_cols.intersection(new_cols))
    
    # Compare values for common records
    old_subset = old_records.loc[common_keys, compare_cols]
    new_subset = new_records.loc[common_keys, compare_cols]
    
    # Check for differences
    differences = []
    for key in common_keys:
        old_row = old_subset.loc[key]
        new_row = new_subset.loc[key]
        
        # Compare each column
        for col in compare_cols:
            old_val = old_row[col]
            new_val = new_row[col]
            
            # Handle NaN comparisons
            if pd.isna(old_val) and pd.isna(new_val):
                continue
            elif pd.isna(old_val) or pd.isna(new_val):
                differences.append({
                    'key': key,
                    'column': col,
                    'old_value': old_val,
                    'new_value': new_val
                })
            elif old_val != new_val:
                differences.append({
                    'key': key,
                    'column': col,
                    'old_value': old_val,
                    'new_value': new_val
                })
    
    if differences:
        print(f"⚠️  WARNING: Found {len(differences)} differences in non-excluded columns!")
        print("\nDifferences found:")
        diff_df = pd.DataFrame(differences)
        print(diff_df.to_string())
    else:
        print(f"✅ Validation passed: {len(common_keys)} updated records match old records (excluding content, views, answer)")
else:
    print("ℹ️  No existing records to update - all records are new")

# Concatenate original df with updated/new records
df = pd.concat([df, expand_to_full_df], ignore_index=True)

# Drop duplicates on merge keys, keeping the last occurrence (from expand_to_full_df)
df = df.drop_duplicates(subset=merge_keys, keep='last', ignore_index=True)

ArrowMemoryError: realloc of size 2607087616 failed

In [ ]:
## Count the records, original count is 530341
df.count()

## The records have been updated, but there will be blank placeholders for 2026.

In [ ]:
## Write to file
# df.to_parquet("./data/final/ChronosAlign_sample_1_update.parquet", index=False)